# 02. Write와 Select

Write는 정보를 외부 기록에 남기는 단계이고, Select는 현재 질문에 필요한 기록만 고르는 단계입니다.

## 실행 환경 준비

실습 파일은 `labs/part2/ch08-context-engineering/notebooks/02-Write와-Select.ipynb`입니다. 새 커널에서 시작해 첫 코드 셀로 패키지 버전을 맞춘 뒤, 설명과 코드를 위에서부터 순서대로 실행합니다. `.env`는 현재 작업 폴더의 상위 경로에서도 찾습니다.

In [1]:
# %pip은 현재 노트북 커널에 패키지를 설치하는 Jupyter 전용 명령입니다.
# -q는 설치 로그를 줄이고, == 뒤 숫자는 수업에서 검증한 버전을 고정합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


## 1. API 연결 준비

각 노트북은 빈 커널에서 시작할 수 있도록 연결 코드를 포함합니다.

In [2]:
# find_dotenv는 현재 폴더부터 상위 폴더를 검색해 .env 경로를 찾습니다.
import os

from dotenv import find_dotenv, load_dotenv

# usecwd=True는 노트북을 실행한 폴더를 검색 시작점으로 지정합니다.
# 키 값은 출력하지 않고 존재 여부만 확인합니다.
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
print(".env와 API 키 확인 완료")

.env와 API 키 확인 완료


### 모델 객체를 만듭니다

`.env` 확인이 끝난 뒤 모델 이름과 출력 제한을 지정합니다. 이 셀은 아직 API에 질문을 보내지 않습니다.

In [3]:
# init_chat_model은 provider:model 형식으로 채팅 모델 객체를 만듭니다.
# model은 이후 셀에서 invoke()를 호출할 때 사용합니다.
from langchain.chat_models import init_chat_model

# openai:는 공급자 이름이며, reasoning_effort="none"은 별도 추론 토큰을 사용하지 않는 설정입니다.
# max_tokens=300은 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model(
    "openai:gpt-5.4-mini",
    reasoning_effort="none",
    max_tokens=300,
)
print("LangChain 모델 준비 완료")

LangChain 모델 준비 완료


## 2. 기록을 JSON으로 저장하고 다시 읽습니다

예제 데이터는 코드 안에 직접 적습니다. `effective_date`는 날짜 필터를 관찰하기 위한 합성 메타데이터이며, PDF의 발행일이나 실제 규정 시행일을 뜻하지 않습니다. 저장 파일은 운영체제의 임시 폴더에 만들기 때문에 노트북을 어느 폴더에서 열어도 실행할 수 있습니다.

In [4]:
# 각 기록에는 선택에 쓸 기관·주제·적용일과 답에 쓸 본문이 있습니다.
import json
from pathlib import Path
from tempfile import gettempdir

records = [
    {"id": "E01-p2", "institution": "미르시립도서관", "topic": "휴관",
     "effective_date": "2025-01-01", "text": "매주 월요일 휴관"},
    {"id": "E01-p3", "institution": "미르시립도서관", "topic": "대출",
     "effective_date": "2025-01-01", "text": "7권, 14일 대출"},
    {"id": "E01-p5", "institution": "미르시립도서관", "topic": "주차",
     "effective_date": "2025-01-01", "text": "회원은 최초 2시간 무료입니다."},
]
print("저장할 기록:", len(records))

저장할 기록: 3


### 임시 파일에 기록을 씁니다

`gettempdir()`를 사용하므로 노트북을 어느 폴더에서 열어도 같은 방식으로 실행됩니다. 출력된 이름으로 저장 파일을 확인합니다.

`Path(gettempdir()) / "ch08-memory.json"`은 임시 폴더에 파일 이름을 연결한 경로입니다. `json.dumps()`는 목록을 JSON 문자열로 바꾸고 `write_text()`가 그 문자열을 저장합니다. 고정 파일 이름이므로 이 셀을 다시 실행하면 같은 이름의 기록을 덮어씁니다. 장기 보존이나 여러 사용자의 저장을 설계한 예제는 아닙니다.

In [5]:
# ensure_ascii=False는 한국어를 그대로 JSON에 기록합니다.
memory_path = Path(gettempdir()) / "ch08-memory.json"
json_text = json.dumps(records, ensure_ascii=False)
memory_path.write_text(json_text, encoding="utf-8")
print("임시 폴더에 저장한 파일:", memory_path.name)

임시 폴더에 저장한 파일: ch08-memory.json


### 파일을 다시 읽어 중간 결과를 확인합니다

`read_text()`는 문자열을 반환하고, `json.loads()`는 그 문자열을 파이썬 목록으로 복원합니다.

In [6]:
# loaded에는 파일에서 복원한 기록 목록이 저장됩니다.
saved_text = memory_path.read_text(encoding="utf-8")
loaded = json.loads(saved_text)
print("다시 읽은 기록:", len(loaded))

다시 읽은 기록: 3


## 3. 질문에 필요한 기록만 선택합니다

기관·주제·질문 날짜를 코드에 적고, 세 조건이 맞는 기록의 ID를 고릅니다. 자동 검색은 Ch 9에서 다룹니다.

`topics`는 필요한 주제의 집합이고 `in`은 포함 여부를 검사합니다. 세 조건을 `and`로 연결하므로 모두 참인 기록만 ID 집합에 넣습니다. 이어지는 반복문은 `loaded`의 원래 순서로 선택한 기록을 모읍니다.

예를 들어 같은 상품의 2025년과 2026년 설명서가 모두 기준일 전에 시행되었다면 두 문서가 함께 선택될 수 있습니다. 아래 조건은 최신 버전 하나를 고르거나 폐기 여부를 확인하지 않습니다. 질문의 기관·주제·날짜 역시 사람이 정한 값이며 모델이 질문에서 자동 추출하지 않습니다.

In [7]:
# criteria는 질문에서 확인한 기관·주제·날짜입니다.
criteria = {
    "institution": "미르시립도서관",
    "topics": {"대출", "주차"},
    "as_of": "2026-09-08",
}
selected_ids = set()
for row in loaded:
    same_institution = row["institution"] == criteria["institution"]
    needed_topic = row["topic"] in criteria["topics"]
    already_effective = row["effective_date"] <= criteria["as_of"]
    if same_institution and needed_topic and already_effective:
        selected_ids.add(row["id"])

selected = []
for row in loaded:
    if row["id"] in selected_ids:
        selected.append(row)

for row in selected:
    print(row["id"], row["topic"], row["text"])

E01-p3 대출 7권, 14일 대출
E01-p5 주차 회원은 최초 2시간 무료입니다.


## 4. 선택한 기록으로 답을 생성합니다

모델에는 저장소 전체 대신 선택한 두 기록만 전달합니다.

In [8]:
# context_lines에 출처 ID와 내용을 한 줄씩 추가합니다.
context_lines = []
for row in selected:
    line = f"[{row['id']}] {row['text']}"
    context_lines.append(line)

context = "\n".join(context_lines)
prompt = f"""근거:
{context}

질문: 대출 권수와 주차 무료 시간을 알려 주세요."""
print(prompt)

근거:
[E01-p3] 7권, 14일 대출
[E01-p5] 회원은 최초 2시간 무료입니다.

질문: 대출 권수와 주차 무료 시간을 알려 주세요.


### 모델에 전달할 메시지를 만듭니다

`system`에는 답변 규칙을, `user`에는 앞에서 만든 질문을 넣습니다. 셀을 실행한 뒤 역할 순서를 확인합니다.

In [9]:
# messages는 역할과 내용을 짝지은 목록입니다.
# system 지시를 user 질문보다 먼저 둡니다.
messages = [
    ("system", "제공된 근거만 사용하고 문장마다 식별자를 붙이세요."),
    ("user", prompt),
]
roles = []
for role, content in messages:
    roles.append(role)
print("메시지 역할:", roles)

메시지 역할: ['system', 'user']


### 모델을 한 번 호출합니다

`invoke()`의 반환값은 `AIMessage`입니다. 사용량 정보가 없으면 이후 비교를 할 수 없으므로 즉시 오류를 냅니다.

In [10]:
# response는 모델의 답과 메타데이터를 가진 AIMessage입니다.
response = model.invoke(messages)
usage = response.usage_metadata

if usage is None:
    raise RuntimeError("모델 사용량 메타데이터가 없습니다.")

### 답과 실제 토큰 수를 관찰합니다

먼저 답변 원문을 읽고, 다음 두 줄에서 API가 보고한 입력·출력 토큰 수를 확인합니다. 답변 문구와 수치는 실행마다 달라질 수 있습니다.

In [11]:
# text는 답변 문자열이고 usage는 실제 호출의 토큰 사용량입니다.
print(response.text)
print("input_tokens:", usage.get("input_tokens"))
print("output_tokens:", usage.get("output_tokens"))

대출 권수는 7권입니다. [E01-p3]  
주차 무료 시간은 최초 2시간입니다. [E01-p5]
input_tokens: 75
output_tokens: 37


## 5. 선택 ID 한 개를 줄여 비교합니다

기본 선택에는 대출과 주차가 들어 있습니다. 주차 ID 하나를 제외한 뒤 선택된 기록 수와 주제를 비교합니다.

In [12]:
# changed_ids에서는 E01-p5 하나만 제외합니다.
changed_ids = selected_ids - {"E01-p5"}
changed_selected = []
for row in loaded:
    if row["id"] in changed_ids:
        changed_selected.append(row)

print("기본 선택 수:", len(selected))
print("변경 선택 수:", len(changed_selected))
print("변경 후 주제:", changed_selected[0]["topic"])

기본 선택 수: 2
변경 선택 수: 1
변경 후 주제: 대출


## 정리

외부 기록은 저장만으로 모델에 전달되지 않습니다. 질문에 맞는 항목을 선택하여 호출 입력에 넣어야 합니다.

## 실패 재현: 저장한 기록을 메시지에 넣지 않습니다

파일에는 세 기록이 있지만 모델에는 빈 근거를 전달합니다. 대출 권수와 주차 시간을 답할 정보가 실제 메시지에 있는지 확인합니다.

In [13]:
# 두 호출에서 질문과 답변 규칙은 그대로 사용합니다.
question = "대출 권수와 주차 무료 시간을 알려 주세요."
answer_rule = "제공된 근거만 사용하고 문장마다 식별자를 붙이세요. 근거가 없으면 모른다고 답하세요."
# 빈 문자열은 저장 파일의 본문을 메시지에 옮기지 않은 상태입니다.
missing_context = ""
before_prompt = f"""근거:
{missing_context}

질문: {question}"""
print(before_prompt)

근거:


질문: 대출 권수와 주차 무료 시간을 알려 주세요.


### 근거가 없는 메시지로 실제 답을 확인합니다

`invoke()`에 파일 경로를 전달하지 않았고, 메시지에도 본문이 없습니다. 모델은 저장 파일의 내용을 자동으로 읽지 않습니다. 출력 문구는 실행마다 달라집니다.

In [14]:
# 근거가 비어 있으면 정확한 권수와 시간을 답할 수 없는지 읽습니다.
before_response = model.invoke([
    ("system", answer_rule),
    ("user", before_prompt),
])
print(before_response.text)

근거가 없어 **모릅니다**.


## 수정: 선택한 기록을 같은 질문의 근거로 넣습니다

앞에서 만든 `context`에는 대출·주차 본문과 출처 ID가 있습니다. 근거 영역만 이 값으로 바꾸어 다시 호출합니다. 답의 7권·2시간과 출처 ID를 원래 기록과 비교합니다.

In [15]:
# context는 앞에서 selected의 본문과 출처 ID를 연결한 문자열입니다.
after_prompt = f"""근거:
{context}

질문: {question}"""
after_response = model.invoke([
    ("system", answer_rule),
    ("user", after_prompt),
])
print("전달한 근거:", context)
print("모델 답:", after_response.text)

전달한 근거: [E01-p3] 7권, 14일 대출
[E01-p5] 회원은 최초 2시간 무료입니다.
모델 답: 대출 권수는 **7권**입니다. [E01-p3]  
주차 무료 시간은 **최초 2시간**입니다. [E01-p5]
